<a href="https://colab.research.google.com/github/PhyloGrok/MolGraphs/blob/master/Network2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pyvis

Import the SUBSET.csv into a bipartite network.

In [ ]:
import pandas as pd
import networkx as nx
from networkx.algorithms import bipartite

# 1. Load your adjacency list file
# Change 'interactions.csv' to your filename. Use sep='\t' if it is a TSV file.
df = pd.read_csv('Adjacency_Filtered.csv')

# 2. Extract unique lists of nodes for each bipartite set
# Assumes your columns are named 'miRNA' and 'mRNA'
mirnas = df['miRNA'].unique()
mrnas = df['Gene'].unique()

# 3. Initialize an empty graph
B = nx.Graph()

# 4. Add nodes with their respective bipartite attribute (0 or 1)
B.add_nodes_from(mirnas, bipartite=0)
B.add_nodes_from(mrnas, bipartite=1)

# 5. Populate edges directly from the DataFrame rows
# This loops through each row and adds an edge between the miRNA and mRNA
edges = list(df.itertuples(index=False, name=None))
B.add_edges_from(edges)

# 6. Verify the import worked correctly
print(f"Graph is bipartite: {bipartite.is_bipartite(B)}")
print(f"Total nodes: {B.number_of_nodes()} ({len(mirnas)} miRNAs, {len(mrnas)} mRNAs)")
print(f"Total interactions: {B.number_of_edges()}")

Graph is bipartite: True
Total nodes: 483 (296 miRNAs, 187 mRNAs)
Total interactions: 1913


:Initial visualization, no loop and no node attribution (Started showing errors?)

In [ ]:
from pyvis.network import Network
#from pyvis.network import Network

# Initialize PyVis network
net = Network(notebook=True, cdn_resources='in_line', height="600px", width="100%", bgcolor="#222222", font_color="white")

# Load your existing NetworkX graph (B)
net.from_nx(B)

# Apply visual configurations based on attributes
for node in net.nodes:
    # 1. Shape/Color based on molecule type, red for miRNA and blue for mRNA
    if node['bipartite'] == 0:
        node['color'] = '#FF6B6B'
        node['shape'] = 'diamond'
    else:
        node['color'] = '#4D96FF'
        node['shape'] = 'dot'

    # 3. Create a hover tooltip showing names and data
    node['title'] = f"ID: {node['id']}<br>Fold Change: {fc:.2f}"

# Turn on physics engine so nodes naturally space themselves out
net.toggle_physics(True)

# Save and display inside the notebook
net.show("mirna_mrna_network_no_exp1.html")

NameError: name 'fc' is not defined

node attribution ONLY MIR155

In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network

# Define your exact experimental condition column
target_condition = 'miR155'

# Load clean pre-filtered mRNA file
mrna_df = pd.read_csv('nanostring_mrna.csv')
# Convert the target_condition column to numeric, coercing errors to NaN
mrna_exp_dict = pd.Series(pd.to_numeric(mrna_df[target_condition], errors='coerce').values, index=mrna_df['ProbeID']).to_dict()

# Load clean pre-filtered miRNA file
# Original code read header as data, which caused 'miR155' string in data. Correct by dropping first row.
mirna_df = pd.read_csv('nanostring_mirna.csv', header=None, names=['ProbeID', 'Accession', 'Analyte', 'Control', 'cel67', 'miR142', 'miR155', 'miR18b', 'miR890'])
mirna_df = mirna_df.drop(0).reset_index(drop=True) # Drop the header row that was read as data
# Convert the target_condition column to numeric, coercing errors to NaN
mirna_exp_dict = pd.Series(pd.to_numeric(mirna_df[target_condition], errors='coerce').values, index=mirna_df['ProbeID']).to_dict()

# Combine both dictionary structures into one tracking database
all_raw_expressions = {**mrna_exp_dict, **mirna_exp_dict}

# Push clean values onto the loaded NetworkX graph (B)
nx.set_node_attributes(B, all_raw_expressions, name='raw_expression')

net = Network(notebook=True, cdn_resources='in_line', height="600px", width="100%", bgcolor="#222222", font_color="white")
net.from_nx(B)

# Calculate independent maximums from the graph layers to preserve visual parity

# Filter out NaN values before calculating max, as NaN cannot be compared.
max_mirna_val = max([val for val in [B.nodes[n].get('raw_expression', 1) for n, attr in B.nodes(data=True) if attr.get('bipartite') == 0] if pd.notna(val)] or [1])
max_mrna_val = max([val for val in [B.nodes[n].get('raw_expression', 1) for n, attr in B.nodes(data=True) if attr.get('bipartite') == 1] if pd.notna(val)] or [1])

for node in net.nodes:
    raw_exp = node.get('raw_expression', 0)

    # Handle potential NaN values that might result from 'coerce' during to_numeric
    if pd.isna(raw_exp):
        raw_exp = 0 # Default to 0 for NaN values

    # Scale each molecule type relative to its own layer maximum (Base size 10 to Max size 40)
    if node['bipartite'] == 0:
        node['color'] = '#FF6B6B'  # Salmon for miRNA
        node['shape'] = 'diamond'
        node['size'] = 5 + ((raw_exp / max_mirna_val) * 35)
    else:
        node['color'] = '#4D96FF'  # Sky Blue for mRNA
        node['shape'] = 'dot'
        node['size'] = 5 + ((raw_exp / max_mrna_val) * 35)

    # Tooltip tracking windows reveal actual absolute numbers on hover
    node['title'] = f"ID: {node['id']}<br>Raw Expression: {raw_exp:.2f}"

#toggle physics for a moving visualization
net.toggle_physics(True)
net.show("mirna_mrna_network_5_35.html")

mirna_mrna_network_5_35.html


LOOP TRIAL FOR CEL67, 142, 155, 18B, 890 **(add log)**, y or x projections, add differentiating colors for plotly visualizations for each? how many miRNAs are regulating, legend centrality (where in the network it is), use the scaling after the log?

find equivalent for network entrality metrics under readme

In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network

target_conditions_list = ['cel67', 'miR142', 'miR155', 'miR18b', 'miR890']

for target_condition in target_conditions_list:
    print(f"Processing for condition: {target_condition}")

    mrna_df = pd.read_csv('nanostring_mrna.csv')

    mrna_exp_dict = pd.Series(pd.to_numeric(mrna_df[target_condition], errors='coerce').values, index=mrna_df['ProbeID']).to_dict()

    mirna_df = pd.read_csv('nanostring_mirna.csv', header=None, names=['ProbeID', 'Accession', 'Analyte', 'Control', 'cel67', 'miR142', 'miR155', 'miR18b', 'miR890'])
    mirna_df = mirna_df.drop(0).reset_index(drop=True)

    mirna_exp_dict = pd.Series(pd.to_numeric(mirna_df[target_condition], errors='coerce').values, index=mirna_df['ProbeID']).to_dict()


    all_raw_expressions = {**mrna_exp_dict, **mirna_exp_dict}

    nx.set_node_attributes(B, all_raw_expressions, name='raw_expression')

    net = Network(notebook=True, cdn_resources='in_line', height="600px", width="100%", bgcolor="#222222", font_color="white")
    net.from_nx(B)


    max_mirna_val = max([val for val in [B.nodes[n].get('raw_expression', 1) for n, attr in B.nodes(data=True) if attr.get('bipartite') == 0] if pd.notna(val)] or [1])
    max_mrna_val = max([val for val in [B.nodes[n].get('raw_expression', 1) for n, attr in B.nodes(data=True) if attr.get('bipartite') == 1] if pd.notna(val)] or [1])

    for node in net.nodes:
        raw_exp = node.get('raw_expression', 0)

        if pd.isna(raw_exp):
            raw_exp = 0

        # Scale each molecule type relative to its own layer maximum (Base size 10 to Max size 40)
        if node['bipartite'] == 0:
            node['color'] = '#FF6B6B'  # Salmon for miRNA
            node['shape'] = 'diamond'
            node['size'] = 5 + ((raw_exp / max_mirna_val) * 35)
        else:
            node['color'] = '#4D96FF'  # Sky Blue for mRNA
            node['shape'] = 'dot'
            node['size'] = 5 + ((raw_exp / max_mrna_val) * 35)

        # Tooltip tracking windows reveal actual absolute numbers on hover
        node['title'] = f"ID: {node['id']}<br>Raw Expression: {raw_exp:.2f}"

    #toggle physics for a moving visualization
    net.toggle_physics(True)
    net.show(f"mirna_mrna_network_{target_condition}.html")


Processing for condition: cel67
mirna_mrna_network_cel67.html
Processing for condition: miR142
mirna_mrna_network_miR142.html
Processing for condition: miR155
mirna_mrna_network_miR155.html
Processing for condition: miR18b
mirna_mrna_network_miR18b.html
Processing for condition: miR890
mirna_mrna_network_miR890.html


PLOTLY LOOP TRIAL,LOG TRANSFORM, WITH LABELS

In [ ]:
#importing
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import numpy as np

#loop to iterate through all conditions + control cel67
target_conditions_list = ['cel67', 'miR142', 'miR155', 'miR18b', 'miR890']

for target_condition in target_conditions_list:
    print(f"Processing for condition: {target_condition}")

#converting values to numbers to map to corresponding probeid, forcing errors into Nan, making expression dictionary (mrna)
    mrna_df = pd.read_csv('nanostring_mrna.csv')
    mrna_exp_dict = pd.Series(pd.to_numeric(mrna_df[target_condition], errors='coerce').values, index=mrna_df['ProbeID']).to_dict()

#dropping header row and making an expression dictionary (mirna)
    mirna_df = pd.read_csv('nanostring_mirna.csv', header=None, names=['ProbeID', 'Accession', 'Analyte', 'Control', 'cel67', 'miR142', 'miR155', 'miR18b', 'miR890'])
    mirna_df = mirna_df.drop(0).reset_index(drop=True)
    mirna_exp_dict = pd.Series(pd.to_numeric(mirna_df[target_condition], errors='coerce').values, index=mirna_df['ProbeID']).to_dict()

#combinging the mirna and mrna expression dictionaries into one --> node attribute called raw expression
    all_raw_expressions = {**mrna_exp_dict, **mirna_exp_dict}
    nx.set_node_attributes(B, all_raw_expressions, name='raw_expression')


#making a 2d plot for every node
    pos = nx.spring_layout(B, k=0.5, iterations=50)

    edge_x = []
    edge_y = []
    for edge in B.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.append(x0)
        edge_x.append(x1)
        edge_x.append(None)
        edge_y.append(y0)
        edge_y.append(y1)
        edge_y.append(None)

#grey line connecting nodes and turning off hover***
    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=0.5, color='#aba6a6'),
        hoverinfo='none',
        mode='lines')
# making lists to hold node configurations to normalize node sizes
#finding the highest expressions for mirna (0) and mrna (1)
    node_x = []
    node_y = []
    node_sizes = []
    node_colors = []
    node_text = []
    node_labels = []

    max_mirna_val = max([val for val in [B.nodes[n].get('raw_expression', 1) for n, attr in B.nodes(data=True) if attr.get('bipartite') == 0] if pd.notna(val)] or [1])
    max_mrna_val = max([val for val in [B.nodes[n].get('raw_expression', 1) for n, attr in B.nodes(data=True) if attr.get('bipartite') == 1] if pd.notna(val)] or [1])

#making 0s (mirnas) red and 1s (mrnas) blue
#using np.log1p to handle expression values equal to 0 and scale expression
    for node_id in B.nodes():
        x, y = pos[node_id]
        node_x.append(x)
        node_y.append(y)

        attr = B.nodes[node_id]
        raw_exp = attr.get('raw_expression', 0)

        if pd.isna(raw_exp):
            raw_exp = 0

        if attr['bipartite'] == 0:
            node_colors.append('#FF6B6B')
            size = 5 + ((np.log1p(raw_exp) / np.log1p(max_mirna_val)) * 35)
        else:
            node_colors.append('#4D96FF')
            size = 5 + ((np.log1p(raw_exp) / np.log1p(max_mrna_val)) * 35)

        node_sizes.append(size)
        node_text.append(f"ID: {node_id}<br>Raw Expression: {raw_exp:.2f}")
        node_labels.append(str(node_id))
#turning on hover and formatting nodes with centered labels in white
#removing axes
    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers + text',
        text=node_labels,
        textposition='middle center',
        hoverinfo='text',
        hovertext=node_text,
        textfont=dict(size = 5, color = 'white'),
        marker=dict(
            showscale=False,
            color=node_colors,
            size=node_sizes,
            line_width=1))

    fig = go.Figure(data=[edge_trace, node_trace],
                 layout=go.Layout(
                    title=f'Network Visualization for {target_condition} (log transform)',
                    titlefont_size=16,
                    showlegend=False,
                    hovermode='closest',
                    margin=dict(b=20,l=5,r=5,t=40),
                    annotations=[ dict(
                        text="Python Plotly Network",
                        showarrow=False,
                        xref="paper", yref="paper",
                        x=0.005, y=-0.002 ) ],
                    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))
    fig.show()


Processing for condition: cel67


Processing for condition: miR142


Processing for condition: miR155


Processing for condition: miR18b


Processing for condition: miR890


eigen centrality and closeness * unfinished

In [ ]:
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import numpy as np

target_conditions_list = ['cel67', 'miR142', 'miR155', 'miR18b', 'miR890']

centrality = []
for target_condition in target_conditions_list:

  mrna_df = pd.read_csv('nanostring_mrna.csv')
  mrna_exp_dict = pd.Series(pd.to_numeric(mrna_df[target_condition], errors = 'coerce').values, index=mrna_df['ProbeID']).to_dict()

  mirna_df = pd.read_csv('nanostring_mirna.csv', header = None, names = ['ProbeID','Accession', 'Analyte', 'Control', 'cel67', 'miR142', 'miR155', 'miR155', 'mir18b', 'miR890'])
  mirna_df = mirna_df.drop(0).reset_index(drop = true)
  mirna_exp_dict = pd.Series(pd.to_numeric(mirna_df[target_condition], errors = 'coerce').values, index = mirna_df['ProbeID']).to_dict()

  all_raw_expressions = {**mrna_exp_dict, **mirna_exp_dict}

  nx.set_node_attributes(B, all_raw_expressions, name = 'raw_expression')

  deg_dict = dict(B.degree())
  bet_dict = nx.betweenness_centrality(B)
  close_dict = nx.closeness_centrality(B)

  eigen_dict = nx.eigenvector_centrality(B)

  centrality.append(deg_dict)
  centrality.append(bet_dict)
  centrality.append(close_dict)
  centrality.append(eigen_dict)

